In [3]:
import pandas as pd

df = pd.read_csv("../data/raw.csv")
print(df.shape)
print(df.dtypes)
df.head(10)

(21837, 6)
waktu                      str
suhu_luar_C            float64
kelembaban_luar_pct    float64
okupansi_orang         float64
hari_libur               int64
konsumsi_kWh           float64
dtype: object


,waktu,suhu_luar_C,kelembaban_luar_pct,okupansi_orang,hari_libur,konsumsi_kWh
0,2024-01-01 00:00,20.9,94.0,0.0,1,88.6
1,2024-01-01 01:00,20.7,85.0,0.0,1,93.2
2,2024-01-01 02:00,19.9,83.0,0.0,1,78.2
3,2024-01-01 03:00,19.3,93.0,0.0,1,77.9
4,2024-01-01 04:00,19.4,83.0,0.0,1,78.3
5,2024-01-01 05:00,19.5,84.0,0.0,1,83.1
6,2024-01-01 06:00,20.9,83.0,0.0,1,85.6
7,2024-01-01 07:00,22.4,80.0,0.0,1,81.3
8,2024-01-01 08:00,23.9,82.0,9.0,1,98.9
9,2024-01-01 09:00,25.8,71.0,8.0,1,91.3


In [4]:
print(df.isna().sum())
df.describe()

waktu                    0
suhu_luar_C             95
kelembaban_luar_pct     95
okupansi_orang         216
hari_libur               0
konsumsi_kWh             0
dtype: int64


,suhu_luar_C,kelembaban_luar_pct,okupansi_orang,hari_libur,konsumsi_kWh
count,21742.000000,21742.000000,21621.000000,21837.000000,21837.000000
mean,24.339803,78.331984,131.242958,0.050556,234.279178
std,4.238748,10.522502,192.950327,0.219095,2945.575999
min,13.600000,45.000000,0.000000,0.000000,-277.000000
25%,20.700000,70.000000,0.000000,0.000000,83.700000
50%,24.300000,78.000000,0.000000,0.000000,90.400000
75%,28.000000,87.000000,351.000000,0.000000,235.400000
max,34.500000,99.000000,581.000000,1.000000,99999.000000


In [5]:
df["waktu"] = pd.to_datetime(df["waktu"], errors="coerce")
print("waktu gagal diparse:", df["waktu"].isna().sum())
print("baris duplikat penuh:", df.duplicated().sum())
print("waktu duplikat:", df["waktu"].duplicated().sum())
print("urutan waktu naik?", df["waktu"].is_monotonic_increasing)
print("rentang:", df["waktu"].min(), "->", df["waktu"].max())

# jam yang hilang
full = pd.date_range(df["waktu"].min(), df["waktu"].max(), freq="h")
missing = full.difference(df["waktu"])
print("jam seharusnya:", len(full), "| jam hilang:", len(missing))
print(missing[:10])

# nilai mustahil di target
k = df["konsumsi_kWh"]
print("konsumsi < 0:", (k < 0).sum())
print("konsumsi == 99999:", (k == 99999).sum())
print("konsumsi == 0:", (k == 0).sum())
print("konsumsi > 500 selain 99999:", ((k > 500) & (k != 99999)).sum())

# apakah kolom kosong terjadi di baris yang sama?
print("suhu & kelembapan kosong bersamaan:",
      (df["suhu_luar_C"].isna() & df["kelembaban_luar_pct"].isna()).sum())

waktu gagal diparse: 747
baris duplikat penuh: 80
waktu duplikat: 821
urutan waktu naik? False
rentang: 2024-01-01 00:00:00 -> 2026-06-30 23:00:00
jam seharusnya: 21888 | jam hilang: 873
DatetimeIndex(['2024-01-10 04:00:00', '2024-01-12 18:00:00',
               '2024-02-03 03:00:00', '2024-02-11 12:00:00',
               '2024-02-18 17:00:00', '2024-03-04 09:00:00',
               '2024-03-07 21:00:00', '2024-03-09 22:00:00',
               '2024-03-14 14:00:00', '2024-04-10 00:00:00'],
              dtype='datetime64[us]', freq=None)
konsumsi < 0: 21
konsumsi == 99999: 19
konsumsi == 0: 41
konsumsi > 500 selain 99999: 0
suhu & kelembapan kosong bersamaan: 95


In [7]:
raw = pd.read_csv("../data/raw.csv", dtype={"waktu": "string"})
parsed = pd.to_datetime(raw["waktu"], format="%Y-%m-%d %H:%M", errors="coerce")
bad = raw.loc[parsed.isna(), "waktu"]

print("gagal parse:", len(bad))
print(bad.sample(25, random_state=0).to_list())

# pola panjang teks, untuk mengelompokkan format
print(bad.str.len().value_counts())

gagal parse: 747
['11/03/2026 15.00', '22/03/2026 15.00', '01/03/2026 14.00', '02/03/2026 07.00', '21/03/2026 23.00', '04/03/2026 05.00', '16/03/2026 20.00', '04/03/2026 01.00', '20/03/2026 13.00', '02/03/2026 03.00', '27/03/2026 14.00', '16/03/2026 05.00', '25/03/2026 05.00', '22/03/2026 16.00', '02/03/2026 14.00', '30/03/2026 19.00', '12/03/2026 16.00', '19/03/2026 23.00', '16/03/2026 23.00', '05/03/2026 15.00', '14/03/2026 23.00', '11/03/2026 07.00', '15/03/2026 00.00', '10/03/2026 11.00', '27/03/2026 04.00']
waktu
16    747
Name: count, dtype: Int64


In [8]:
raw = pd.read_csv("../data/raw.csv", dtype={"waktu": "string"})

t1 = pd.to_datetime(raw["waktu"], format="%Y-%m-%d %H:%M", errors="coerce")
t2 = pd.to_datetime(raw["waktu"], format="%d/%m/%Y %H.%M", errors="coerce")
raw["waktu"] = t1.fillna(t2)

print("masih gagal parse:", raw["waktu"].isna().sum())
print("baris format DD/MM:", t2.notna().sum())
print("rentang format DD/MM:", t2.min(), "->", t2.max())

# hitung ulang
print("baris duplikat penuh:", raw.duplicated().sum())
print("waktu duplikat:", raw["waktu"].duplicated().sum())
print("urutan waktu naik?", raw["waktu"].is_monotonic_increasing)

full = pd.date_range("2024-01-01 00:00", "2026-06-30 23:00", freq="h")
missing = full.difference(raw["waktu"])
print("jam seharusnya:", len(full), "| jam hilang:", len(missing))

masih gagal parse: 0
baris format DD/MM: 747
rentang format DD/MM: 2026-03-01 00:00:00 -> 2026-03-31 23:00:00
baris duplikat penuh: 80
waktu duplikat: 80
urutan waktu naik? True
jam seharusnya: 21888 | jam hilang: 131


In [9]:
full = pd.date_range("2024-01-01 00:00", "2026-06-30 23:00", freq="h")

d = raw.drop_duplicates().set_index("waktu").sort_index().reindex(full)
d.index.name = "waktu"
print(d.shape)   # harus (21888, 5)

# nol: kapan terjadinya?
z = d[d["konsumsi_kWh"] == 0]
print("nol per jam:\n", z.index.hour.value_counts().sort_index().to_string())
print("nol per bulan:\n", z.index.to_period("M").value_counts().sort_index().to_string())

# tandai nilai mustahil sebagai NaN (nol belum disentuh)
k = d["konsumsi_kWh"]
d["konsumsi_kWh"] = k.mask((k < 0) | (k == 99999))

# panjang celah berurutan pada target
def gap_lengths(s):
    na = s.isna()
    grp = (na != na.shift()).cumsum()
    runs = na.groupby(grp).agg(["first", "size"])
    return runs.loc[runs["first"], "size"].value_counts().sort_index()

print("celah konsumsi (panjang: jumlah):\n", gap_lengths(d["konsumsi_kWh"]).to_string())
print("celah okupansi:\n", gap_lengths(d["okupansi_orang"]).to_string())
print("celah suhu:\n", gap_lengths(d["suhu_luar_C"]).to_string())

# sanity check kolom lain
print(d["hari_libur"].value_counts(dropna=False))

(21888, 5)
nol per jam:
 waktu
0     2
2     3
3     2
4     1
6     2
7     1
8     1
9     2
10    2
11    1
12    4
13    1
15    2
16    2
17    4
18    3
19    1
20    2
21    1
22    2
23    2
nol per bulan:
 waktu
2024-01    1
2024-02    1
2024-03    1
2024-04    2
2024-06    2
2024-08    2
2024-09    1
2024-10    3
2024-11    1
2024-12    2
2025-01    1
2025-02    2
2025-03    1
2025-05    3
2025-06    1
2025-07    2
2025-08    1
2025-09    1
2025-11    2
2025-12    2
2026-01    1
2026-02    2
2026-03    4
2026-05    1
2026-06    1
Freq: M
celah konsumsi (panjang: jumlah):
 size
1    171
celah okupansi:
 size
1    341
2      3
celah suhu:
 size
1     129
97      1
hari_libur
0.0    20658
1.0     1099
NaN      131
Name: count, dtype: int64


In [10]:
# 0. di mana celah suhu 97 jam itu?
na = d["suhu_luar_C"].isna()
run_id = (na != na.shift()).cumsum()
run_len = na.groupby(run_id).transform("sum")
long_idx = d.index[na & (run_len > 3)]
print("celah panjang suhu:", long_idx.min(), "->", long_idx.max(), "|", len(long_idx), "jam")

# 1. konsumsi: nol -> NaN, simpan penanda dan versi terisi (untuk fitur lag saja)
d["konsumsi_kWh"] = d["konsumsi_kWh"].mask(d["konsumsi_kWh"] == 0)
d["target_valid"] = d["konsumsi_kWh"].notna()
d["konsumsi_filled"] = d["konsumsi_kWh"].interpolate(method="time", limit=3)

# 2. hari_libur: konstan per tanggal
d["hari_libur"] = d.groupby(d.index.date)["hari_libur"].transform("max").astype(int)

# 3. okupansi: celah pendek
d["okupansi_orang"] = d["okupansi_orang"].interpolate(method="time", limit=2).round()

# 4. suhu dan kelembapan: celah pendek diinterpolasi, celah panjang diisi profil bulan-jam
for c in ["suhu_luar_C", "kelembaban_luar_pct"]:
    s = d[c]
    na = s.isna()
    run_id = (na != na.shift()).cumsum()
    long_gap = na & (na.groupby(run_id).transform("sum") > 3)
    prof = s.groupby([d.index.month, d.index.hour]).transform("mean")
    d[c] = s.interpolate(method="time").mask(long_gap, prof)

# 5. cek hasil
print(d.isna().sum())
print("baris target valid:", d["target_valid"].sum(), "dari", len(d))
print(d.describe().T[["min", "mean", "max"]])

# 6. simpan
d.reset_index().to_csv("../data/clean.csv", index=False)

celah panjang suhu: 2025-10-07 00:00:00 -> 2025-10-11 00:00:00 | 97 jam
suhu_luar_C              0
kelembaban_luar_pct      0
okupansi_orang           0
hari_libur               0
konsumsi_kWh           212
target_valid             0
konsumsi_filled          0
dtype: int64
baris target valid: 21676 dari 21888
                      min        mean    max
suhu_luar_C          13.6   24.345187   34.5
kelembaban_luar_pct  45.0   78.324970   99.0
okupansi_orang        0.0  131.183388  581.0
hari_libur            0.0    0.050439    1.0
konsumsi_kWh         62.6  147.943966  398.5
konsumsi_filled      62.6  147.950215  398.5
